# 🗺️ Model 06 — Station Geo Router
**AirVintage Production ML Pipeline**

| Property | Value |
|----------|-------|
| Dataset | `stations_cleaned.csv` |
| Algorithm | **KNN (k=3, haversine) + City/State Name Matching** |
| Task | Station Discovery & Intelligent Model Routing |
| Use Case | *"Find nearest station to this city"* → Route to correct model |

---
**Role in AirVintage:**  
The Geo Router is the **gateway** of the entire inference pipeline. When a production query arrives (e.g., user's GPS location or typed city name), this model:
1. **Finds the closest active monitoring station** via city/state name matching or KNN on geocoded coordinates
2. **Selects the appropriate model** (city_day, station_day, city_hour, station_hour, or weather_aware)
3. **Returns station metadata** for the response (station name, city, state)

This enables fully automatic, zero-configuration AQI queries from the frontend.

In [ ]:
import sys, os, warnings, json
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('.'))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.neighbors import BallTree
from sklearn.preprocessing import LabelEncoder

DATASET    = 'stations_cleaned.csv'
MODEL_KEY  = 'geo_router'
MODELS_DIR = 'models'
REGISTRY   = f'{MODELS_DIR}/model_registry.json'
os.makedirs(MODELS_DIR, exist_ok=True)
print('✅ Environment ready')

In [ ]:
# ── Load Station Metadata
df = pd.read_csv(DATASET)
print(f'Shape  : {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'States : {df["State"].nunique()}')
print(f'Cities : {df["City"].nunique()}')
print(f'Status : {df["Status"].value_counts().to_dict()}')
df.head(10)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  EDA — Station Network Analysis                          ║
# ╚══════════════════════════════════════════════════════════╝
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Stations per state
state_counts = df['State'].value_counts().head(15)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(state_counts)))
axes[0,0].barh(state_counts.index[::-1], state_counts.values[::-1], color=colors)
axes[0,0].set_title('Stations per State (Top 15)', fontweight='bold')
axes[0,0].set_xlabel('Number of Stations')

# Stations per city
city_counts = df['City'].value_counts().head(10)
axes[0,1].bar(city_counts.index, city_counts.values,
              color=plt.cm.plasma(np.linspace(0.2,0.9,len(city_counts))), edgecolor='white')
axes[0,1].set_title('Cities with Most Stations (Top 10)', fontweight='bold')
axes[0,1].set_xlabel('City'); axes[0,1].tick_params(axis='x', rotation=40)
for bar, val in zip(axes[0,1].patches, city_counts.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                   str(val), ha='center', va='bottom', fontsize=9)

# Status distribution
status = df['Status'].value_counts()
axes[1,0].pie(status.values, labels=status.index, autopct='%1.1f%%',
              colors=['#2ecc71','#e74c3c','#f1c40f','#95a5a6'][:len(status)], startangle=90)
axes[1,0].set_title('Station Status Distribution', fontweight='bold')

# State × City matrix (heatmap)
top_states = df['State'].value_counts().head(8).index
sub = df[df['State'].isin(top_states)]
pivot = sub.groupby(['State'])['City'].nunique()
axes[1,1].bar(pivot.index, pivot.values,
              color=plt.cm.cool(np.linspace(0.3,0.9,len(pivot))), edgecolor='white')
axes[1,1].set_title('Unique Cities Monitored per State (Top 8)', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=35)

fig.suptitle('AirVintage Station Network — Geographic Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/eda_06_station_geo.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  BUILD CITY/STATE LOOKUP INDEX                           ║
# ╚══════════════════════════════════════════════════════════╝

# Build multi-level lookup dictionary
# Priority: City match → State match → Fallback (most common station)

df['City_clean']  = df['City'].str.strip().str.lower()
df['State_clean'] = df['State'].str.strip().str.lower()
df['StationName_clean'] = df['StationName'].str.strip().str.lower()

# Index 1: City → [list of StationIds]
city_to_stations = df.groupby('City_clean')['StationId'].apply(list).to_dict()

# Index 2: State → [list of StationIds]
state_to_stations = df.groupby('State_clean')['StationId'].apply(list).to_dict()

# Index 3: Active only city lookup
active_df = df[df['Status'] == 'Active'].copy()
active_city_to_stations = active_df.groupby('City_clean')['StationId'].apply(list).to_dict()
active_state_to_stations = active_df.groupby('State_clean')['StationId'].apply(list).to_dict()

# Index 4: StationId → full metadata
station_metadata = df.set_index('StationId')[['StationName','City','State','Status']].to_dict(orient='index')

print(f'✅ City index   : {len(city_to_stations)} cities')
print(f'✅ State index  : {len(state_to_stations)} states')
print(f'✅ Active cities: {len(active_city_to_stations)}')
print(f'✅ Station meta : {len(station_metadata)} stations')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  BUILD KNN ON GEOCODED CITY COORDINATES                  ║
# ║  Using a curated lat/lon for major Indian cities         ║
# ╚══════════════════════════════════════════════════════════╝

# Major Indian city coordinates (curated — offline, no API needed)
CITY_COORDS = {
    'delhi': (28.6139, 77.2090), 'mumbai': (19.0760, 72.8777),
    'bangalore': (12.9716, 77.5946), 'bengaluru': (12.9716, 77.5946),
    'chennai': (13.0827, 80.2707), 'kolkata': (22.5726, 88.3639),
    'hyderabad': (17.3850, 78.4867), 'pune': (18.5204, 73.8567),
    'ahmedabad': (23.0225, 72.5714), 'jaipur': (26.9124, 75.7873),
    'surat': (21.1702, 72.8311), 'lucknow': (26.8467, 80.9462),
    'kanpur': (26.4499, 80.3319), 'nagpur': (21.1458, 79.0882),
    'bhopal': (23.2599, 77.4126), 'visakhapatnam': (17.6868, 83.2185),
    'patna': (25.5941, 85.1376), 'vadodara': (22.3072, 73.1812),
    'ludhiana': (30.9010, 75.8573), 'agra': (27.1767, 78.0081),
    'nashik': (20.0059, 73.7897), 'faridabad': (28.4089, 77.3178),
    'meerut': (28.9845, 77.7064), 'rajkot': (22.3039, 70.8022),
    'kalyan': (19.2403, 73.1305), 'vasai': (19.3919, 72.8397),
    'varanasi': (25.3176, 82.9739), 'srinagar': (34.0837, 74.7973),
    'aurangabad': (19.8762, 75.3433), 'amritsar': (31.6340, 74.8723),
    'ranchi': (23.3441, 85.3096), 'guwahati': (26.1445, 91.7362),
    'chandigarh': (30.7333, 76.7794), 'coimbatore': (11.0168, 76.9558),
    'kolhapur': (16.7050, 74.2433), 'thiruvananthapuram': (8.5241, 76.9366),
    'kochi': (9.9312, 76.2673), 'indore': (22.7196, 75.8577),
    'gwalior': (26.2183, 78.1828), 'vijayawada': (16.5062, 80.6480),
    'jodhpur': (26.2389, 73.0243), 'raipur': (21.2514, 81.6296),
    'kota': (25.2138, 75.8648), 'ghaziabad': (28.6692, 77.4538),
    'amaravati': (16.5731, 80.3642), 'tirupati': (13.6288, 79.4192),
    'rajamahendravaram': (17.0005, 81.8040), 'guntur': (16.3067, 80.4365),
    'solapur': (17.6599, 75.9064), 'hubli': (15.3647, 75.1240),
}

# Which cities in our station dataset have known coords?
city_list = df['City_clean'].unique()
matched   = [c for c in city_list if c in CITY_COORDS]
print(f'Cities with known coords: {len(matched)} / {len(city_list)}')

In [ ]:
# Build KNN BallTree on known city coordinates
df['lat'] = df['City_clean'].map(lambda c: CITY_COORDS.get(c, (None, None))[0])
df['lon'] = df['City_clean'].map(lambda c: CITY_COORDS.get(c, (None, None))[1])

df_geo = df.dropna(subset=['lat','lon']).copy()
coords_rad = np.deg2rad(df_geo[['lat','lon']].values)

# BallTree with haversine metric (exact great-circle distance)
ball_tree = BallTree(coords_rad, metric='haversine')

print(f'✅ BallTree built on {len(df_geo)} stations with geocoordinates')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  GEO ROUTER FUNCTIONS                                    ║
# ╚══════════════════════════════════════════════════════════╝

def route_by_city(city: str, prefer_active: bool = True) -> dict:
    """
    Find monitoring stations for a city query.
    Returns the best matching station and routing metadata.
    """
    city_q = city.strip().lower()
    
    # Tier 1: Exact city match (active first)
    if prefer_active and city_q in active_city_to_stations:
        station_ids = active_city_to_stations[city_q]
        matched_by  = 'city_exact_active'
    elif city_q in city_to_stations:
        station_ids = city_to_stations[city_q]
        matched_by  = 'city_exact'
    else:
        # Tier 2: Fuzzy city match (substring)
        fuzzy = [c for c in city_to_stations if city_q in c or c in city_q]
        if fuzzy:
            station_ids = city_to_stations[fuzzy[0]]
            matched_by  = f'city_fuzzy:{fuzzy[0]}'
        else:
            station_ids = [df['StationId'].iloc[0]]  # ultimate fallback
            matched_by  = 'fallback'
    
    primary_station = station_ids[0]
    meta = station_metadata.get(primary_station, {})
    
    return {
        'query_city'    : city,
        'station_id'    : primary_station,
        'station_name'  : meta.get('StationName','Unknown'),
        'city'          : meta.get('City','Unknown'),
        'state'         : meta.get('State','Unknown'),
        'status'        : meta.get('Status','Unknown'),
        'all_stations'  : station_ids,
        'matched_by'    : matched_by,
        'recommended_model': 'station_day',
    }


def route_by_latlon(lat: float, lon: float, k: int = 3) -> dict:
    """
    Find nearest k monitoring stations to a GPS coordinate.
    Uses BallTree haversine distance.
    """
    query = np.deg2rad([[lat, lon]])
    dists, indices = ball_tree.query(query, k=min(k, len(df_geo)))
    
    R_km = 6371.0  # Earth radius in km
    results = []
    for dist, idx in zip(dists[0], indices[0]):
        row   = df_geo.iloc[idx]
        dist_km = dist * R_km
        results.append({
            'station_id'  : row['StationId'],
            'station_name': row['StationName'],
            'city'        : row['City'],
            'state'       : row['State'],
            'status'      : row['Status'],
            'distance_km' : round(dist_km, 2),
        })
    
    primary = results[0]
    return {
        'query_lat'        : lat,
        'query_lon'        : lon,
        'nearest_station'  : primary['station_id'],
        'station_name'     : primary['station_name'],
        'city'             : primary['city'],
        'state'            : primary['state'],
        'distance_km'      : primary['distance_km'],
        'k_nearest'        : results,
        'recommended_model': 'station_hour',
    }


def get_model_recommendation(query_type: str, has_weather: bool = False) -> str:
    """
    Recommend the best AirVintage model based on query context.
    
    Parameters
    ----------
    query_type  : 'city_daily', 'city_hourly', 'station_daily', 'station_hourly'
    has_weather : If True and weather data available, prefer weather model
    """
    if has_weather:
        return 'air_weather'
    routing = {
        'city_daily'    : 'city_day',
        'city_hourly'   : 'city_hour',
        'station_daily' : 'station_day',
        'station_hourly': 'station_hour',
    }
    return routing.get(query_type, 'city_day')


print('✅ Geo Router functions defined')

In [ ]:
# ── Validation Tests
print('=== Geo Router Validation Tests ===')
print()

# Test 1: City match
r1 = route_by_city('Delhi')
print(f'Test 1 — City "Delhi":')
print(f'  Station: {r1["station_name"][:50]}  |  Matched by: {r1["matched_by"]}')

# Test 2: Lat/lon
r2 = route_by_latlon(28.6139, 77.2090, k=3)  # Delhi coordinates
print(f'\nTest 2 — Lat/Lon (28.6139, 77.2090):')
for nn in r2['k_nearest']:
    print(f'  {nn["station_id"]:8s} | {nn["city"]:20s} | {nn["distance_km"]:.1f} km')

# Test 3: Fuzzy city match
r3 = route_by_city('Bengaluru')
print(f'\nTest 3 — City "Bengaluru" (fuzzy):')
print(f'  Station: {r3["station_name"][:50]}  |  Matched by: {r3["matched_by"]}')

# Test 4: Model recommendation
for qt in ['city_daily','city_hourly','station_daily','station_hourly']:
    print(f'  recommend({qt:16s}) → {get_model_recommendation(qt)}')

In [ ]:
# ── Station Coverage Map (matplotlib)
fig, ax = plt.subplots(figsize=(14, 10))

# India bounding box (approximate)
ax.set_xlim(67, 98); ax.set_ylim(6, 38)

# Plot stations with geocoordinates
status_colors = {'Active':'#2ecc71', 'Unknown':'#f1c40f', 'Inactive':'#e74c3c'}
for status, grp in df_geo.groupby('Status'):
    ax.scatter(grp['lon'], grp['lat'],
               c=status_colors.get(status,'#95a5a6'),
               s=60, alpha=0.75, label=status, edgecolor='white', linewidth=0.5, zorder=3)

# Annotate major cities
major = {'Delhi':(77.2090,28.6139),'Mumbai':(72.8777,19.0760),
         'Kolkata':(88.3639,22.5726),'Chennai':(80.2707,13.0827),
         'Bangalore':(77.5946,12.9716),'Hyderabad':(78.4867,17.3850)}
for city, (lon, lat) in major.items():
    ax.annotate(city, (lon, lat), xytext=(5,5), textcoords='offset points',
                fontsize=9, fontweight='bold', color='#2c3e50')

ax.set_title('AirVintage Monitoring Station Network — India', fontsize=15, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(title='Station Status', loc='lower right')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/map_06_station_network.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Station network map saved')

In [ ]:
# ── Save All Router Artifacts
router_artifacts = {
    'city_to_stations'       : city_to_stations,
    'state_to_stations'      : state_to_stations,
    'active_city_to_stations': active_city_to_stations,
    'active_state_to_stations': active_state_to_stations,
    'station_metadata'       : station_metadata,
    'city_coords'            : CITY_COORDS,
}

with open(f'{MODELS_DIR}/06_geo_router_index.json','w') as f:
    json.dump(router_artifacts, f, indent=2)

# Save BallTree
joblib.dump(ball_tree, f'{MODELS_DIR}/06_geo_router_balltree.pkl')

# Save geocoded station dataframe
df_geo[['StationId','StationName','City','State','Status','lat','lon']].to_csv(
    f'{MODELS_DIR}/06_geo_station_coords.csv', index=False)

# Save station index for quick lookup
df[['StationId','StationName','City','State','Status','City_clean','State_clean']].to_csv(
    f'{MODELS_DIR}/06_station_index.csv', index=False)

# Update registry
if os.path.exists(REGISTRY):
    with open(REGISTRY,'r') as f: reg = json.load(f)
else:
    reg = {}

reg['geo_router'] = {
    'algorithm'   : 'BallTree KNN (haversine) + City Name Matching',
    'dataset'     : DATASET,
    'stations'    : len(df),
    'geocoded'    : len(df_geo),
    'model_paths' : {
        'index'    : f'{MODELS_DIR}/06_geo_router_index.json',
        'balltree' : f'{MODELS_DIR}/06_geo_router_balltree.pkl',
        'coords'   : f'{MODELS_DIR}/06_geo_station_coords.csv',
        'station_index': f'{MODELS_DIR}/06_station_index.csv',
    },
    'version' : '1.0.0',
    'notes'   : 'Routes queries to correct model. City name matching + KNN haversine on geocoords.'
}

with open(REGISTRY,'w') as f:
    json.dump(reg, f, indent=2)

print('✅ All Geo Router artifacts saved')
print(f'   {MODELS_DIR}/06_geo_router_index.json')
print(f'   {MODELS_DIR}/06_geo_router_balltree.pkl')
print(f'   {MODELS_DIR}/06_geo_station_coords.csv')
print(f'   {MODELS_DIR}/06_station_index.csv')

---
## ✅ Model 06 — Geo Router Complete

This router is the **production gateway** used by `inference_router.py`.

### Routing Decision Tree
```
Incoming Query
    ├── Has lat/lon?    → BallTree KNN → Nearest station → station_hour (with weather: air_weather)
    ├── Has city name?  → City lookup  → Station list    → city_day / city_hour
    └── Has station ID? → Direct route → station_day / station_hour
```

### Complete AirVintage Model Registry

| Model | Key | Algorithm |
|-------|-----|-----------|
| 01 | `city_day` | XGBoost |
| 02 | `station_day` | RF + LightGBM Ensemble |
| 03 | `city_hour` | LightGBM |
| 04 | `station_hour` | CatBoost |
| 05 | `air_weather` | XGBoost + Weather Features |
| 06 | `geo_router` | BallTree KNN + City Matching |
